In [14]:
# import library
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [15]:
# load dataset 
df=pd.read_csv('fraud_detection_stratified_kfold_practice.csv')
df

,transaction_amount,transaction_hour,account_age_days,failed_attempts,international,is_fraud
0,432.69,5,238,2,1,1
1,126.59,4,83,1,1,1
2,237.67,10,569,0,0,0
3,161.73,23,283,2,1,1
4,41.60,18,71,0,0,0
...,...,...,...,...,...,...
995,25.06,12,1280,0,0,0
996,44.06,18,1783,0,0,0
997,46.82,22,96,2,1,1
998,223.81,2,487,1,0,1


In [16]:
# check daataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_amount  1000 non-null   float64
 1   transaction_hour    1000 non-null   int64  
 2   account_age_days    1000 non-null   int64  
 3   failed_attempts     1000 non-null   int64  
 4   international       1000 non-null   int64  
 5   is_fraud            1000 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 47.0 KB


In [17]:
# check missing value 
print("missing value:",df.isna().sum())
print("-"*50)
# check duplocate 
print("duplicated:",df.duplicated().sum())

missing value: transaction_amount    0
transaction_hour      0
account_age_days      0
failed_attempts       0
international         0
is_fraud              0
dtype: int64
--------------------------------------------------
duplicated: 0


In [18]:
df.describe()

,transaction_amount,transaction_hour,account_age_days,failed_attempts,international,is_fraud
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000
mean,74.003680,12.916000,1407.006000,0.406000,0.171000,0.10000
std,95.813926,5.436037,908.914008,0.764034,0.376697,0.30015
min,5.230000,0.000000,4.000000,0.000000,0.000000,0.00000
25%,28.757500,9.000000,538.250000,0.000000,0.000000,0.00000
50%,48.005000,13.000000,1389.500000,0.000000,0.000000,0.00000
75%,80.862500,17.250000,2200.250000,1.000000,0.000000,0.00000
max,1171.380000,23.000000,2999.000000,6.000000,1.000000,1.00000


In [19]:
# divide data into x and y
X=df.drop(columns='is_fraud')
y=df['is_fraud']

In [20]:
print("x:",X.shape)
print("y:",y.shape)

x: (1000, 5)
y: (1000,)


In [21]:
# make a pipeline for scaling and logistic regression
pipeline=Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression())
])

In [24]:
scor=cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring='accuracy'
)
print("cross validation score:",scor*100)

cross validation score: [96.5 99.  97.5 97.  96. ]


In [30]:
# predict the model using stratified kfold
from sklearn.model_selection import cross_val_predict


y_pred=cross_val_predict(
    pipeline,
    X,
    y,
    cv=5
)
print("accuracy score:",accuracy_score(y,y_pred)*100)
print("classification report:\n",classification_report(y,y_pred))


accuracy score: 97.2
classification report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98       900
           1       0.89      0.82      0.85       100

    accuracy                           0.97      1000
   macro avg       0.94      0.90      0.92      1000
weighted avg       0.97      0.97      0.97      1000



In [29]:
# in simple train test split we can have imbalanced data in train
X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# create  logistic regression
logistic = LogisticRegression()
logistic.fit(X_train,y_train)
logistic_pred=logistic.predict(X_test)

clss=classification_report(y_test,logistic_pred)
accuracy=accuracy_score(y_test,logistic_pred)
print("classification report:\n",clss)
print("accuracy:\n",accuracy*100)

classification report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       180
           1       0.95      0.90      0.92        20

    accuracy                           0.98       200
   macro avg       0.97      0.95      0.96       200
weighted avg       0.98      0.98      0.98       200

accuracy:
 98.5
